# 09 — Monitoramento do modelo
**Hackathon Databricks · Terra Signal · Grupo 4**

**Equipe:** André Ferreira, Antonio Menezes, Júlia Vale e Lucas Menghi

**O que este notebook faz:** acompanha o modelo depois da entrega. A cada nova carteira pontuada, ele compara o que o
modelo recebe e o que ele devolve com o que ele viu no treino, acende um semáforo por indicador e grava as tabelas que
o aplicativo de monitoramento (`app_monitoramento`, em Streamlit) mostra.

**Por que monitorar:** um modelo de churn envelhece. Se a empresa muda preço, lança um plano ou muda a base de clientes,
as relações que o modelo aprendeu deixam de valer, e as previsões pioram sem que ninguém perceba. O monitoramento avisa
antes de o resultado de negócio mostrar o problema.

| # | Pergunta de negócio | Etapa |
|---|---|---|
| Mo1 | A carteira que o modelo está pontuando é parecida com a do treino? | 2. Deriva das variáveis (PSI) |
| Mo2 | O modelo está devolvendo o que devolvia? | 2. Taxa de "Yes" e distribuição das notas |
| Mo3 | O modelo em produção é o mesmo que foi avaliado? | 1.2 Reprodução pelo MLflow |
| Mo4 | O monitor acende quando a carteira muda? | 3. Lote simulado com deriva |
| Mo5 | Quando retreinar? | 6. Regras de retreino |

| Entrada | Notebook | O que traz |
|---|---|---|
| `gold_churn_abt_scored` | 06 | A referência: os clientes do histórico, com a marcação treino/teste e as notas |
| `gold_churn_inferencia_scored` | 06 | O lote atual: a carteira pontuada |
| `gold_churn_segmentos` | 07 | O segmento e o quadrante de cada cliente |
| `gold_churn_modelo_info` | 06 | O modelo, o limiar e as métricas do teste |

| Saída | Para quê |
|---|---|
| `gold_churn_monitoramento` | Um indicador por linha, com valor, referência, limites e semáforo, por lote |
| `gold_churn_monitoramento_distribuicao` | A distribuição de cada variável, referência × lote (gráficos do app) |
| `gold_churn_fila_retencao` | A fila de clientes do lote atual, com nota, segmento e ação |
| `app_monitoramento/dados/*.csv` | A mesma informação em arquivo, para o app rodar sem SQL Warehouse |

In [ ]:
%run ./_comum

## Glossário

| Termo | Por extenso | O que é | Onde aparece |
|---|---|---|---|
| **Lote** | — | Uma carteira de clientes pontuada pelo modelo numa data (aqui, os 1.409 da inferência) | Todo o notebook |
| **Referência** | — | O que o modelo viu ao aprender: os clientes do treino (variáveis) e do teste (notas) | Seção 1 |
| **Deriva** | *Drift* | Mudança na carteira ou nas notas em relação à referência | Seção 2 |
| **PSI** | *Population Stability Index* (índice de estabilidade da população) | Mede o quanto a distribuição de uma variável mudou: soma de (% lote − % referência) × ln(% lote ÷ % referência) em cada faixa | Seção 2 |
| **Semáforo** | — | Estável (verde), atenção (amarelo), alerta (vermelho) ou aguardando (cinza), por indicador | Seções 2 e 4 |
| **MLflow** | — | O registro do modelo no Databricks; a 1.2 carrega o modelo de lá e confere as notas | 1.2 |
| **Recall** | Sensibilidade | De quem saiu de fato, quantos o modelo marcou; só dá para medir quando o churn real chegar | Seção 2 |

## Checklist de decisões

**A mesma tabela em todos os notebooks.** ☑ = decidido (com a data); ◐ = sugestão nossa, a confirmar (as de variáveis, na modelagem); ☐ = em aberto.

| # | Decisão | Opções | Proposta | Impacto | Evidência no 01 | No 02 | Depende de | Decisão do grupo |
|---|---|---|---|---|---|---|---|---|
| D1 | Grafia de `PhoneService` (70 + 23) | Padronizar | Padronizar para `Yes`/`No` | Sem isso, `yes` e `Yes` viram categorias diferentes no modelo | 4.1 | Aplicado | Carlos: só erro de padronização (respondido) | ☑ Padronizar para `Yes`/`No` (10/11) |
| D2 | `TotalCharges` nulo (278 + 64) | a) preencher com `MonthlyCharges` × `tenure`; b) excluir; c) deixar ausente | a | Excluir não serve para os 64 da inferência, que precisam de previsão, e perde o sinal da 4.3. A regra vale para 99% dos clientes | 4.1, 4.2, 4.3 | Aplicado | G2; Carlos: preencher assim é "razoável, desde que lembrem que é uma aproximação" (respondido); falta saber por que o valor está ausente | ☑ Preencher com `MonthlyCharges` × `tenure`, como aproximação (10/11) |
| D3 | `TotalCharges` nulo com `tenure` = 0 (10 + 1) | a) 0; b) outro valor | a (sai da própria regra: mensalidade × 0) | Clientes novos, sem cobrança ainda; efeito pequeno | 4.2 | Aplicado | G2 | ☑ 0 (10/11) |
| D4 | `tenure` = `unknown` (19 + 4) | a) `TotalCharges` ÷ `MonthlyCharges`, arredondado; b) mediana; c) excluir | a; mediana do histórico no único cliente sem as duas colunas | `tenure` é das variáveis mais ligadas ao churn: um erro aqui muda a faixa de tempo de casa do cliente | 4.1, 4.2 | Aplicado | G2; Carlos: 72 meses é teto real (respondido) | ☑ `TotalCharges` ÷ `MonthlyCharges`, arredondado e limitado a 72; mediana do histórico no caso sem os dois (10/11) |
| D5 | Indicador `fl_total_charges_ausente` | Criar ou não | Criar e comparar o modelo com e sem ele | Preserva o sinal da ausência (33,1% contra 26,4% de churn, p = 0,016) | 4.3 | Criado | Carlos: por que falta `TotalCharges` | ☑ Criar; o modelo testa com e sem (10/11) |
| D6 | Indicador `fl_tenure_ausente` | Criar ou não | Criar, mas tende a sair na seleção (só 19 casos, p = 0,21) | Pouca informação; uma coluna a mais | 4.3 | Não criado | G2 | ☑ Não criar: 19 casos, p = 0,21 (10/11) |
| D7 | `TotalCharges` fora de ±20% da regra (47 + 9) | a) manter; b) investigar; c) excluir | a | Mostra que o preenchimento de D2 é uma aproximação | 4.2 | Mantidos | Carlos: o preço muda com plano e promoções, e não há histórico de preços (respondido) | ☑ Manter (10/11) |
| D8 | Quebras de linha no `CustomerFeedback` (85 + 15) | Remover ou manter | Remover, se o texto for usado | Só importa se o texto entrar na análise ou no modelo | 4.2 | Aplicado | D9 | ☑ Trocar a quebra de linha por espaço (10/11) |
| D9 | `CustomerFeedback` no modelo | a) com; b) sem; c) os dois | c, com o modelo sem texto como principal | Possível vazamento: uma regra sobre o texto acerta 98,9% | Célula 2 (seção 1); 7.3 do 03 | Fora do 02 (modelo) | G1; Carlos: feedback antes ou depois do cancelamento | ☑ Fora da ABT. O texto vai na tabela separada `gold_churn_texto`, para o modelo com texto (contraexemplo); o modelo oficial é sem texto (10/11) |
| D10 | Serviços com 3 categorias ou sim/não | a) manter 3; b) binário | Testar as duas | "No internet service" repete `InternetService` em 6 colunas: redundância | 3 | Fora do 02 (notebook 04) | G4 | ◐ Sugestão: serviços como sim/não (`_bin`), sempre junto com `InternetService`. A ABT traz as duas versões; seleção final na modelagem (10/11) |
| D11 | Fonte e schema do grupo | a) CSVs de `dados/`; b) tabelas `bronze_*` em `terra_signal` | b | Todos os notebooks; os números precisam bater entre os integrantes | 1.2, 2 | Aplicado: o 02 lê das tabelas `bronze_*` | G3; Carlos: CSV com quebra de linha, schema | ☑ Tabelas `bronze_*` em `workspace.terra_signal`; ignorar (sem apagar) as do upload em `workspace.default` (10/11) |
| D12 | `MonthlyIncome` no modelo | a) manter como candidata; b) ignorar | a, e deixar a seleção decidir | Base externa, sem garantia de precisão; os organizadores dizem que pode ser ignorada. Na exploratória, a renda não muda o risco | 3; seção 10 do 03 | Convertida para número e mantida | Carlos: renda familiar em R$, base externa (respondido) | ◐ Sugestão: descartar (IV abaixo de 0,02, base externa). Fica na ABT; seleção final na modelagem (10/11) |
| D13 | Linhas do `Prediction.csv` | a) 1.409 (clientes); b) 1.461 (linhas físicas) | a | Arquivo com o número errado de linhas pode invalidar a entrega | 1.2 | Fora do 02 (entrega) | Organização: confirmar | ☑ 1.409, uma por `customerID`; a confirmar com a organização (10/11) |
| D14 | Formato da gold | — | `gold_churn_treino` (5.634, com `Churn` 1/0) e `gold_churn_inferencia` (1.409, sem `Churn`), mesmas colunas e ordem; números como número, categorias como texto; comentário em cada coluna; `gold_churn_previsoes` depois do modelo | É o que a modelagem usa; formato estável evita retrabalho | — | Fora do 02 (notebook 04) | — | ☑ ABT com todas as candidatas (originais limpas, serviços nas duas versões, indicador e derivadas) + dicionário com a sugestão de cada coluna; texto em `gold_churn_texto` (10/11) |
| D15 | Outras variáveis sem poder preditivo | a) descartar já; b) deixar a seleção decidir | a: `gender`, `PhoneService` e `MultipleLines` | IV abaixo de 0,02; modelo mais simples de explicar; sexo é variável sensível | Seção 7 e 8 do 03 | Fora do 02 (notebook 04) | — | ◐ Sugestão: descartar `gender`, `PhoneService` e `MultipleLines` (IV abaixo de 0,02). Ficam na ABT; seleção final na modelagem (10/11) |
| D16 | `MonthlyCharges` e `TotalCharges` (repetem outras variáveis) | a) manter; b) trocar por derivada; c) deixar a seleção decidir | Decidir no 04, com as derivadas | Repetem `InternetService` e `tenure` (associação ≥ 0,8) | Seção 6 e 8 do 03 | Fora do 02 (notebook 04) | 04 | ◐ Sugestão: manter `MonthlyCharges` (ou a mensalidade relativa ao pacote) e descartar `TotalCharges` (repete `tenure`, ρ 0,89). As três ficam na ABT; seleção final na modelagem (10/11) |
| D17 | Fonte do 03 | a) CSVs; b) tabelas `bronze_*` | b | Coerência com a D11; os números não mudam | — | Fora do 02 (notebook 03) | — | ☑ Ler da bronze (10/11) |
| D18 | Divisão de papéis | — | A preparação entrega a ABT e sugere; a seleção final das variáveis é feita na modelagem | Quem decide a seleção final das variáveis | — | — | — | ☑ Como proposto (10/11) |
| D19 | Seleção de variáveis | a) só as recomendadas; b) recomendadas + seleção para frente por AUC; c) testes estatísticos com correção para múltiplos testes e retirada das duplicatas | c, conferida com b | É o que entra no modelo | — | — | Notebook 05 | ☑ As 41 colunas do notebook 05 (seções 2 e 3): saem 6 categóricas redundantes e 12 variáveis (duplicatas exatas, versões repetidas do tempo de casa, `gender` sem associação após a correção FDR). A conferência com os métodos da aula (11 variáveis, AUC 0,841) cabe dentro das 41 (11/11) |
| D20 | Segmentação de clientes | a) sem segmentação; b) K-means no comportamento, sem o churn, cruzado com risco × valor | b | Diz que tipo de cliente está em risco e que oferta faz sentido | — | — | Notebook 07 | ◐ Sugestão para o negócio e a apresentação (11/11) |
| D21 | Modelo final | a) logística simples; b) regularização (Lasso, Ridge, Elastic Net); c) árvores (*random forest*, XGBoost, CatBoost) | O de melhor desempenho e mais consistente entre treino, teste e base total | É o modelo que vai para a entrega | — | — | Notebook 06 | ☑ Notebook 06 (modelo do grupo): logística Lasso, 41 colunas, C = 0,3, classes balanceadas, vencedora pela regra de consistência (Ridge a 0,003); AUC 0,830 no teste. Atenção: o balanceamento infla a probabilidade (média de 41% para 27% de churn real) (11/11) |
| D22 | Ponto de corte do Sim/Não | a) 0,5; b) o que maximiza o F1; c) pelo custo da ação | b | Define quantos clientes viram "Yes" | — | — | Notebook 06 | ☑ 0,5046: maior F1 nas probabilidades fora da dobra do treino (notebook 06) (11/11) |
| D23 | Arquivo da entrega | — | 1.409 linhas, `customerID` e `prediction` (Yes/No), na ordem do `inference.csv`, no formato do notebook oficial | Arquivo fora do formato pode invalidar a entrega | — | — | D13; notebook 08 | ☑ `prediction_grupo4_<data_hora>.csv` gerado no 08 e conferido nas 8 regras: 571 de 1.409 com Yes (40,5%); falta comparar os nomes das colunas com o notebook oficial (11/11) |
| D24 | Monitoramento do modelo | a) sem monitoramento; b) PSI das variáveis e das notas, taxa de Yes e reprodução pelo MLflow, com semáforo e app | b | Avisa quando o modelo envelhece, antes de o resultado de negócio mostrar | — | — | Notebook 09 | ◐ Proposta: notebook 09 e app em Streamlit (11/11) |
| D25 | Motor de atendimento | a) só a fila de retenção; b) a nota, os motivos e a oferta de cada cliente na hora da ligação, e a comparação de pacotes para quem ainda não é cliente | b | Leva o modelo para quem fala com o cliente | — | — | Notebook 10 | ◐ Proposta: notebook 10 e página Atendimento do app (12/11) |

## 1. Carregar a referência e o lote

**O que faço:** leio a ABT pontuada (referência), a carteira pontuada (lote atual), os segmentos e o resumo do modelo.
Depois confiro se o modelo registrado no MLflow devolve as mesmas notas que foram publicadas (Mo3).

In [ ]:
# 1.1 Carrega a referência e o lote atual
LOTE = "carteira_2026_11"            # nome do lote pontuado; trocar a cada nova carteira
assert "FQ" in globals(), "rode antes a célula %run ./_comum"
abt = spark.table(f"{FQ}.gold_churn_abt_scored").toPandas()
lote = spark.table(f"{FQ}.gold_churn_inferencia_scored").toPandas()
seg = spark.table(f"{FQ}.gold_churn_segmentos").toPandas()
info = spark.table(f"{FQ}.gold_churn_modelo_info").toPandas().iloc[0]
LIMIAR = float(info["limiar"])

ref_var = abt[abt["Base_Modelagem"] == "Treino"].merge(                 # referência das variáveis: o treino
    seg.loc[seg["base"] == "treino", ["customerID", "segmento"]], on="customerID", how="left", validate="one_to_one")
ref_nota = abt[abt["Base_Modelagem"] == "Teste"].copy()                 # referência das notas: o teste
lote = lote.merge(seg.loc[seg["base"] == "inferencia", ["customerID", "segmento", "nome_segmento", "quadrante", "acao_sugerida"]],
                  on="customerID", how="left", validate="one_to_one")

print(f"Modelo: {info['modelo']} · limiar {virg(LIMIAR, 4)}")
print(f"Referência: {virg(len(ref_var), 0)} clientes do treino (variáveis) e {virg(len(ref_nota), 0)} do teste (notas)")
print(f"Lote {LOTE}: {virg(len(lote), 0)} clientes, {virg(int(lote['Churn_Previsto'].sum()), 0)} previstos como Yes")

In [ ]:
# 1.2 O modelo registrado no MLflow reproduz as notas publicadas? (Mo3)
# endereço do registro gravado pelo 06 (6.4) no resumo do modelo; o ID fixo é o da execução de 11/11
MODEL_URI = info.get("model_uri") if isinstance(info.get("model_uri"), str) else "models:/m-2dd8f256aa384a98bf46d77f84ff80eb"
print("Modelo no MLflow:", MODEL_URI)
try:
    import mlflow
    pipe = mlflow.sklearn.load_model(MODEL_URI)
    inf_bruta = spark.table(f"{FQ}.gold_churn_inferencia").toPandas()
    p_mlflow = pd.Series(pipe.predict_proba(inf_bruta)[:, 1], index=inf_bruta["customerID"])
    diferenca = (p_mlflow - lote.set_index("customerID")["Probabilidade_Churn"]).abs().max()
    REPRODUZ = bool(diferenca < 1e-9)
    print(f"Maior diferença entre o MLflow e a tabela publicada: {diferenca:.2e} → reproduz: {REPRODUZ}")
except Exception as erro:
    REPRODUZ = None
    print("Não foi possível carregar o modelo do MLflow:", type(erro).__name__, str(erro)[:200])

> **Como ler (1.2):** a diferença máxima precisa ser praticamente zero. Se o MLflow não carregar (por permissão
> ou versão), o indicador fica em "atenção", e o resto do monitoramento continua.

## 2. Os indicadores e o semáforo

> **Mo1 — A carteira que o modelo está pontuando é parecida com a do treino?** · **Mo2 — O modelo está devolvendo o que devolvia?**

**Teoria: o PSI.** Divido a variável em faixas (os decis da referência, nas contínuas; as categorias, nas categóricas)
e comparo a proporção de clientes em cada faixa:

> **PSI = Σ (% lote − % referência) × ln(% lote ÷ % referência)**

Se as proporções forem iguais, o PSI é zero. É a mesma medida que o 03 usou para comparar o histórico com a inferência
(máximo de 0,011).

| PSI | Leitura usual no mercado | Semáforo |
|---|---|---|
| Abaixo de 0,10 | Sem mudança relevante | Estável |
| De 0,10 a 0,25 | Mudança moderada: investigar | Atenção |
| Acima de 0,25 | Mudança forte: o modelo pode estar desatualizado | Alerta |

**Indicadores da saída do modelo:**

| Indicador | Referência | Atenção | Alerta |
|---|---|---|---|
| Taxa de "Yes" | A do teste (cerca de 40%) | Diferença de 5 a 10 pontos percentuais | Mais de 10 pontos |
| PSI das notas | As notas do teste (clientes que o modelo não viu, como a carteira) | 0,10 a 0,25 | Acima de 0,25 |
| Reprodução pelo MLflow | As notas publicadas na `gold_churn_inferencia_scored` | Não foi possível carregar | Notas diferentes |
| Recall e precisão reais | Os do teste (78% e 52%) | Queda de 5 a 10 pontos | Queda de mais de 10 pontos |

**Por que a referência das notas é o teste, e não o treino:** no treino, o modelo já viu os clientes, e as notas
saem mais extremas. A carteira, como o teste, é de clientes que o modelo não viu.

**Variáveis monitoradas:** as do núcleo do modelo e do negócio (contrato, tempo de casa, internet, forma de pagamento,
mensalidade, suporte técnico, segurança online, idoso) e a mistura de segmentos.

In [ ]:
# 2.1 Funções: distribuição por faixa, PSI e semáforo
VARIAVEIS = {                                  # variável: tipo (contínua vira faixas pelos decis da referência)
    "Contract": "categorica", "tenure": "continua", "InternetService": "categorica", "PaymentMethod": "categorica",
    "MonthlyCharges": "continua", "TechSupport_bin": "categorica", "OnlineSecurity_bin": "categorica",
    "SeniorCitizen": "categorica", "segmento": "categorica",
}
NOME_VAR = {"Contract": "Tipo de contrato", "tenure": "Tempo de casa (meses)", "InternetService": "Tipo de internet",
            "PaymentMethod": "Forma de pagamento", "MonthlyCharges": "Mensalidade (R$)", "TechSupport_bin": "Suporte técnico",
            "OnlineSecurity_bin": "Segurança online", "SeniorCitizen": "Idoso", "segmento": "Segmento (07)",
            "Probabilidade_Churn": "Nota do modelo"}
LIM_PSI = (0.10, 0.25)
LIM_PP = (5.0, 10.0)                           # pontos percentuais


def faixas(ref, alvo, tipo, n=10):
    """Proporção de clientes por faixa (decis da referência) ou por categoria, na referência e no alvo."""
    if tipo == "continua":
        cortes = np.unique(np.quantile(ref.dropna(), np.linspace(0, 1, n + 1)))
        cortes[0], cortes[-1] = -np.inf, np.inf
        nomes = []
        for a, b in zip(cortes[:-1], cortes[1:]):
            nomes.append(f"até {virg(b, 1)}" if np.isinf(a) else f"acima de {virg(a, 1)}" if np.isinf(b)
                         else f"{virg(a, 1)} a {virg(b, 1)}")
        fr = pd.cut(ref, cortes, labels=nomes)
        fa = pd.cut(alvo, cortes, labels=nomes)
        ordem = nomes
    else:
        fr, fa = ref.astype(str), alvo.astype(str)
        if set(fr) | set(fa) <= {"0", "1", "0.0", "1.0"}:                    # variáveis 0/1: rótulos legíveis
            sim_nao = {"0": "Não", "1": "Sim", "0.0": "Não", "1.0": "Sim"}
            fr, fa = fr.map(sim_nao), fa.map(sim_nao)
        ordem = sorted(set(fr) | set(fa))
    tab = pd.DataFrame({"pct_referencia": fr.value_counts(normalize=True).reindex(ordem, fill_value=0).to_numpy(),
                        "pct_lote": fa.value_counts(normalize=True).reindex(ordem, fill_value=0).to_numpy()},
                       index=pd.Index(ordem, name="faixa"))
    return tab


def psi(tab, eps=1e-4):
    r = np.clip(tab["pct_referencia"].to_numpy(), eps, None)
    a = np.clip(tab["pct_lote"].to_numpy(), eps, None)
    return float(((a - r) * np.log(a / r)).sum())


def semaforo(valor, limites):
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return "aguardando"
    return "estável" if valor < limites[0] else "atenção" if valor < limites[1] else "alerta"

In [ ]:
# 2.2 Calcula os indicadores de um lote
def monitorar(lote_df, nome_lote, reproduz):
    agora = datetime.now().strftime("%Y-%m-%d %H:%M")
    linhas, dist = [], []

    def reg(grupo, indicador, valor, referencia, unidade, limites, status, leitura):
        linhas.append({"data_execucao": agora, "lote": nome_lote, "grupo": grupo, "indicador": indicador,
                       "valor": valor, "referencia": referencia, "unidade": unidade,
                       "limite_atencao": limites[0] if limites else None, "limite_alerta": limites[1] if limites else None,
                       "status": status, "leitura": leitura})

    # saída do modelo
    taxa, taxa_ref = 100 * lote_df["Churn_Previsto"].mean(), 100 * ref_nota["Churn_Previsto"].mean()
    reg("Saída do modelo", "Taxa de Yes", taxa, taxa_ref, "%", LIM_PP, semaforo(abs(taxa - taxa_ref), LIM_PP),
        f"{virg(taxa, 1)}% de Yes no lote, contra {virg(taxa_ref, 1)}% no teste")
    t = faixas(ref_nota["Probabilidade_Churn"], lote_df["Probabilidade_Churn"], "continua")
    v = psi(t)
    reg("Saída do modelo", "PSI das notas", v, 0.0, "PSI", LIM_PSI, semaforo(v, LIM_PSI),
        "Distribuição das notas do lote contra a do teste")
    dist.append(t.reset_index().assign(variavel="Probabilidade_Churn"))
    st_rep = "estável" if reproduz is True else "alerta" if reproduz is False else "atenção"
    reg("Saída do modelo", "Reprodução pelo MLflow", float(reproduz) if reproduz is not None else None, 1.0, "sim/não", None,
        st_rep, {True: "O modelo registrado devolve as mesmas notas publicadas", False: "As notas publicadas diferem do modelo registrado",
                 None: "Não foi possível carregar o modelo do MLflow nesta execução"}[reproduz])
    reg("Desempenho real", "Recall real", None, 100 * float(info["recall_teste"]), "%", LIM_PP, "aguardando",
        "Precisa do churn real dos clientes do lote (por exemplo, 30 dias depois)")
    reg("Desempenho real", "Precisão real", None, 100 * float(info["precision_teste"]), "%", LIM_PP, "aguardando",
        "Precisa do churn real dos clientes do lote")

    # entrada do modelo
    for var, tipo in VARIAVEIS.items():
        t = faixas(ref_var[var], lote_df[var], tipo)
        v = psi(t)
        maior = (t["pct_lote"] - t["pct_referencia"]).abs().idxmax()
        dif = 100 * (t.loc[maior, "pct_lote"] - t.loc[maior, "pct_referencia"])
        reg("Entrada do modelo", NOME_VAR[var], v, 0.0, "PSI", LIM_PSI, semaforo(v, LIM_PSI),
            f"Maior mudança: {maior} ({'+' if dif >= 0 else '−'}{virg(abs(dif), 1)} p.p.)")
        dist.append(t.reset_index().assign(variavel=var))

    # qualidade dos dados
    ausentes = int(lote_df[list(VARIAVEIS)].isna().sum().sum())
    reg("Qualidade dos dados", "Valores ausentes", float(ausentes), 0.0, "células", (1, 1),
        "estável" if ausentes == 0 else "alerta", "Valores ausentes nas variáveis monitoradas")
    dup = int(lote_df["customerID"].duplicated().sum())
    reg("Qualidade dos dados", "Clientes repetidos", float(dup), 0.0, "clientes", (1, 1),
        "estável" if dup == 0 else "alerta", "customerID repetido no lote")

    ind = pd.DataFrame(linhas)
    dist = pd.concat(dist, ignore_index=True)
    dist.insert(0, "lote", nome_lote)
    dist["ordem"] = dist.groupby("variavel").cumcount()
    return ind, dist


from datetime import datetime
ind_lote, dist_lote = monitorar(lote, LOTE, REPRODUZ)
display(ind_lote[["grupo", "indicador", "valor", "referencia", "status", "leitura"]].round(3))

> **Como ler e respostas (Mo1 a Mo3)**
> - **Cada linha é um indicador,** com o valor no lote, a referência e o semáforo. O PSI de cada variável compara a
>   carteira com o treino; o das notas compara com o teste.
> - **O esperado para esta carteira:** tudo estável. O 03 já tinha mostrado PSI máximo de 0,011 entre histórico e
>   inferência, e o 06 mostrou a mesma taxa de Yes (cerca de 40%) no treino, no teste e na inferência.
> - **Recall e precisão reais ficam "aguardando":** só dá para medir quando a empresa souber quem dos 1.409 saiu.
>   A 2.3 mostra como calcular quando esse dado chegar.

In [ ]:
# 2.3 Quando o churn real chegar: recall e precisão no lote (rodar só com a coluna Churn_Real preenchida)
def avaliar_desempenho(lote_df, coluna_real="Churn_Real"):
    """Devolve recall e precisão reais do lote, em %, para comparar com os do teste (indicadores "Desempenho real")."""
    real, prev = lote_df[coluna_real].astype(int), lote_df["Churn_Previsto"].astype(int)
    vp = int(((real == 1) & (prev == 1)).sum())
    recall = 100 * vp / max(int(real.sum()), 1)
    precisao = 100 * vp / max(int(prev.sum()), 1)
    return {"Recall real": recall, "Precisão real": precisao}

# Exemplo de uso, quando houver o resultado (não roda agora):
# resultado = spark.table(f"{FQ}.churn_real_carteira_2026_11").toPandas()       # customerID, Churn_Real
# reais = avaliar_desempenho(lote.merge(resultado, on="customerID"))
# for nome, valor in reais.items():
#     ref = ind_lote.loc[ind_lote["indicador"] == nome, "referencia"].iloc[0]
#     ind_lote.loc[ind_lote["indicador"] == nome, ["valor", "status"]] = [valor, semaforo(ref - valor, LIM_PP)]
print("Função avaliar_desempenho pronta; aguardando o churn real do lote.")

## 3. O monitor acende quando a carteira muda?

> **Mo4 — O monitor acende quando a carteira muda?**

**O que faço:** um alarme que nunca toca pode estar quebrado. Para testar, monto um **lote simulado**: a mesma
carteira, sorteada de novo com reposição, dando **peso 3 aos clientes de fibra no contrato mensal** (o segmento S1, o
de maior risco). É o que aconteceria se uma campanha de fibra trouxesse muitos clientes novos no plano mensal.

**O esperado:** contrato, internet e mensalidade mudam, a taxa de Yes sobe e o semáforo passa para atenção ou alerta.
O lote simulado vai para as mesmas tabelas, com o nome `simulacao_fibra_mensal`, para o app mostrar os dois lado a lado.

In [ ]:
# 3.1 Lote simulado: peso 3 para fibra no contrato mensal
rng = np.random.default_rng(42)
peso = np.where((lote["Contract"] == "Month-to-month") & (lote["InternetService"] == "Fiber optic"), 3.0, 1.0)
idx = rng.choice(lote.index.to_numpy(), size=len(lote), replace=True, p=peso / peso.sum())
lote_sim = lote.loc[idx].reset_index(drop=True)
lote_sim["customerID"] = [f"{c}-sim{i}" for i, c in enumerate(lote_sim["customerID"])]   # sorteio com reposição: novos ids
LOTE_SIM = "simulacao_fibra_mensal"

ind_sim, dist_sim = monitorar(lote_sim, LOTE_SIM, REPRODUZ)
comparar = (ind_lote.set_index("indicador")[["valor", "status"]]
            .join(ind_sim.set_index("indicador")[["valor", "status"]], lsuffix="_carteira", rsuffix="_simulado"))
display(comparar.round(3))
print("Fibra no contrato mensal: "
      f"{virg(100 * ((lote['Contract'] == 'Month-to-month') & (lote['InternetService'] == 'Fiber optic')).mean(), 1)}% na carteira, "
      f"{virg(100 * ((lote_sim['Contract'] == 'Month-to-month') & (lote_sim['InternetService'] == 'Fiber optic')).mean(), 1)}% no simulado")

> **Como ler e resposta (Mo4)**
> - **Coluna "carteira" × "simulado":** o mesmo indicador nos dois lotes. Na carteira real, o esperado é tudo estável;
>   no simulado, contrato, internet, mensalidade e segmento mudam, e a taxa de Yes sobe.
> - **Resposta:** se o simulado acende o semáforo e a carteira não, o monitor separa uma carteira estável de uma que
>   mudou. É a prova de que o alarme funciona antes de precisar dele.

## 4. O monitor em um gráfico

In [ ]:
# 4.1 Gráfico: o PSI de cada variável, carteira × simulado (gráfico de halteres)
psi_tab = (ind_lote[ind_lote["unidade"] == "PSI"].set_index("indicador")["valor"].rename("carteira").to_frame()
           .join(ind_sim[ind_sim["unidade"] == "PSI"].set_index("indicador")["valor"].rename("simulado")))
psi_tab = psi_tab.sort_values("simulado")
fig, ax = plt.subplots(figsize=(10, 5.2))
yy = np.arange(len(psi_tab))
for lim, rot in [(LIM_PSI[0], "atenção"), (LIM_PSI[1], "alerta")]:
    ax.axvline(lim, color="#cccccc", linestyle=":", linewidth=1, zorder=0)
    ax.text(lim, len(psi_tab) - 0.35, rot, ha="center", fontsize=8.5, color="#888888")
for y0, (_, r) in zip(yy, psi_tab.iterrows()):
    ax.plot([r["carteira"], r["simulado"]], [y0, y0], color="#dddddd", linewidth=2.5, zorder=1)
ax.scatter(psi_tab["carteira"], yy, color=CINZA, s=55, zorder=2)
ax.scatter(psi_tab["simulado"], yy, color=[DESTAQUE if v >= LIM_PSI[0] else COR for v in psi_tab["simulado"]], s=70, zorder=3)
for y0, (_, r) in zip(yy, psi_tab.iterrows()):
    ax.text(r["simulado"] + 0.006, y0, virg(r["simulado"], 3), va="center", fontsize=8.5, color=TEXTO)
ax.text(psi_tab["carteira"].iloc[-1], len(psi_tab) - 1 + 0.38, "carteira real", ha="center", fontsize=8.5, color="#777777")
ax.text(psi_tab["simulado"].iloc[-1], len(psi_tab) - 1 + 0.38, "lote simulado", ha="center", fontsize=8.5, color=DESTAQUE)
ax.set_yticks(yy, psi_tab.index, fontsize=9.5)
ax.set_xlabel("PSI (índice de estabilidade da população): 0 = igual à referência")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: virg(v, 2)))
ax.set_xlim(0, max(psi_tab.max().max() * 1.25, LIM_PSI[1] * 1.15))
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
n_alerta = int((psi_tab["simulado"] >= LIM_PSI[0]).sum())
fig.suptitle(f"Na carteira real, nenhuma variável mudou; no lote simulado, {n_alerta} passam do limite de atenção:\n"
             "o monitor separa uma carteira estável de uma que mudou",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
rodape(fig, ["Contract", "tenure", "InternetService", "PaymentMethod", "MonthlyCharges"],
       fonte="gold_churn_abt_scored (referência), gold_churn_inferencia_scored (carteira) e lote simulado")
plt.show()

> **Como ler o gráfico (4.1)**
> - **Onde olhar:** a distância entre o ponto cinza (carteira real) e o colorido (lote simulado) em cada linha.
> - **O que observar:** os pontos que passam da linha pontilhada de atenção (0,10) ou de alerta (0,25).
> - **O que comparar:** as variáveis ligadas à simulação (contrato, internet, mensalidade, segmento) com as que não
>   deveriam mudar (idoso, por exemplo).
> - **Achado:** a carteira real fica toda perto de zero; o lote simulado acende justamente nas variáveis que mudamos.

## 5. Gravar as tabelas e os arquivos do aplicativo

**O que faço:** gravo os indicadores e as distribuições dos dois lotes, e a fila de retenção do lote real. Se o lote
já existe na tabela, ele é substituído (rodar de novo não duplica). Também salvo os mesmos dados em CSV na pasta
`app_monitoramento/dados`, para o aplicativo rodar mesmo sem acesso a um SQL Warehouse.

In [ ]:
# 5.1 Grava gold_churn_monitoramento, gold_churn_monitoramento_distribuicao e gold_churn_fila_retencao
def gravar_historico(nome, novo):
    """Substitui as linhas dos lotes de agora e mantém os lotes anteriores (histórico)."""
    if spark.catalog.tableExists(f"{FQ}.{nome}"):                  # na primeira execução a tabela ainda não existe
        antigo = spark.table(f"{FQ}.{nome}").toPandas()
        antigo = antigo[~antigo["lote"].isin(novo["lote"].unique())]
        tudo = pd.concat([antigo, novo], ignore_index=True)
    else:
        tudo = novo
    spark.createDataFrame(tudo).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.{nome}")
    return tudo

monit = gravar_historico("gold_churn_monitoramento", pd.concat([ind_lote, ind_sim], ignore_index=True))
distr = gravar_historico("gold_churn_monitoramento_distribuicao", pd.concat([dist_lote, dist_sim], ignore_index=True))

COLS_FILA = ["customerID", "Probabilidade_Churn", "Churn_Previsto", "Decil_Risco", "segmento", "nome_segmento", "quadrante",
             "acao_sugerida", "MonthlyCharges", "Contract", "tenure", "InternetService", "PaymentMethod"]
fila = lote[COLS_FILA].sort_values("Probabilidade_Churn", ascending=False).assign(lote=LOTE)
spark.createDataFrame(fila).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ}.gold_churn_fila_retencao")

for nome, desc in [("gold_churn_monitoramento", "Indicadores do monitoramento do modelo de churn, um por linha e por lote (notebook 09)"),
                   ("gold_churn_monitoramento_distribuicao", "Distribuição das variáveis monitoradas, referência x lote (notebook 09)"),
                   ("gold_churn_fila_retencao", "Fila de retenção do lote atual: nota, segmento, quadrante e ação (notebook 09)")]:
    spark.sql(f"COMMENT ON TABLE {FQ}.{nome} IS '{desc}'")
    print(f"{FQ}.{nome}: {virg(spark.table(f'{FQ}.{nome}').count(), 0)} linhas")

In [ ]:
# 5.2 Salva os mesmos dados em CSV, na pasta do aplicativo (para rodar sem SQL Warehouse)
PASTA_APP = os.path.join(raiz, "app_monitoramento", "dados")
os.makedirs(PASTA_APP, exist_ok=True)
perfil = spark.table(f"{FQ}.gold_churn_segmentos_perfil").toPandas()
for nome, df in [("gold_churn_monitoramento", monit), ("gold_churn_monitoramento_distribuicao", distr),
                 ("gold_churn_fila_retencao", fila), ("gold_churn_modelo_info", info.to_frame().T),
                 ("gold_churn_segmentos_perfil", perfil)]:
    caminho = os.path.join(PASTA_APP, f"{nome}.csv")
    df.to_csv(caminho, index=False)
    print(f"{caminho}: {virg(len(df), 0)} linhas")

## 6. Quando retreinar

> **Mo5 — Quando retreinar?**

| Sinal | Onde aparece | O que fazer |
|---|---|---|
| Alguma variável do núcleo (contrato, tempo de casa, internet, pagamento) em **alerta** (PSI acima de 0,25) | 2.2 e app, página Deriva | Retreinar: rodar 04, 05 e 06 com o histórico atualizado |
| Taxa de Yes **mais de 10 pontos** longe da do teste | 2.2 e app, página Saúde | Investigar a carteira; se a deriva se confirmar, retreinar |
| **Recall real** mais de 10 pontos abaixo do teste (78%) | 2.3, quando o churn real chegar | Retreinar e rever o limiar |
| Reprodução pelo MLflow em **alerta** | 1.2 | Não usar as notas: o modelo em produção não é o avaliado. Publicar de novo pelo 06 |
| Indicadores em **atenção** | 2.2 | Acompanhar no lote seguinte; registrar o motivo provável |

**Como agendar no Databricks:** em *Jobs & Pipelines*, criar um *job* com uma tarefa deste notebook, agendada para
logo depois de cada nova pontuação (por exemplo, todo dia 1º do mês). O app lê as tabelas que ele grava e se atualiza
sozinho.

## Checklist final do 09

| # | Decisão | Status | Onde |
|---|---|---|---|
| D24 | Monitoramento do modelo | ◐ PSI das variáveis e das notas, taxa de Yes, reprodução pelo MLflow e qualidade dos dados, com semáforo; lote simulado como teste do alarme; app em Streamlit (11/11) | Seções 1 a 6 |

**Saída do 09:** `gold_churn_monitoramento`, `gold_churn_monitoramento_distribuicao`, `gold_churn_fila_retencao` e os
CSV em `app_monitoramento/dados`.

**Próximo passo:** publicar o aplicativo `app_monitoramento` como Databricks App (instruções no `README.md` da pasta).